# SurveyResponder Google Colab Sweep
This notebook automatically sets up Ollama, clones the repository, and runs the LLM survey sweep. Just click "Run All"!

In [ ]:
# 1. Setup Environment
!git clone https://github.com/DhruvKithany/SurveyResponder.git
%cd SurveyResponder
!pip install -r requirements.txt

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# 2. Configuration
# Change this to any model you want (e.g., qwen2.5:7b, qwen2.5:14b, llama3.1:8b)
MODEL_NAME = "qwen2.5:7b"

# The temperatures to sweep across
TEMPERATURES = [1.65, 1.75, 1.85, 2.00]

# Number of synthetic respondents per temperature
NUM_RESPONSES = 50


In [ ]:
# 3. Run Ollama & Execute Sweep
import subprocess
import time
import sys
import os

# Start Ollama server in the background
print("Starting Ollama server...")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Give it a few seconds to initialize

# Pull the model
print(f"Pulling {MODEL_NAME}...")
!ollama pull {MODEL_NAME}

# Create output directory
base_dir = f"{MODEL_NAME.replace(':', '_')}_Runs"
os.makedirs(base_dir, exist_ok=True)

print("Starting sweep...")
for temp in TEMPERATURES:
    temp_dir = os.path.join(base_dir, f"temp_{temp:.2f}")
    os.makedirs(temp_dir, exist_ok=True)
    output_file = os.path.join(temp_dir, "results.csv")
    log_file_path = os.path.join(temp_dir, "run_log.txt")
    
    cmd = [
        sys.executable, "cli.py", "run",
        "--questions", "prca_questions.json",
        "--persona", "persona.json",
        "--model", MODEL_NAME,
        "--num-responses", str(NUM_RESPONSES),
        "--output", output_file,
        "--temperature", str(temp)
    ]
    
    print(f"\nRunning for temperature {temp:.2f}...")
    # We stream the output to the notebook directly instead of a background log file
    result = subprocess.run(cmd)
    
    if result.returncode == 0:
        print(f"Completed run for temperature {temp:.2f}.")
    else:
        print(f"Error running for temperature {temp:.2f}.")
        break

print("\nAll runs finished! Check the generated CSV files in the sidebar.")
